In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ISIn import BurstDetector

## Helper Functions

In [2]:
def filter_bursts_by_min_involved_electrodes(burst_time_intervals, df_recording, min_electrodes=4):
    filtered_bursts = []
    for burst in burst_time_intervals:
        df_filtered = df_recording[(df_recording["Time"] >= burst[0]) & (df_recording["Time"] <= burst[1])]
        num_electrodes = df_filtered["Channel"].nunique()
        if num_electrodes >= min_electrodes:
            filtered_bursts.append(burst)
    return filtered_bursts

<img src="./assets/features_fig.png" alt="Features Figure" width="600" height="300">

Reference: [https://analysis-pipeline.readthedocs.io/en/latest/meanap-methods.html](https://analysis-pipeline.readthedocs.io/en/latest/meanap-methods.html)

In [3]:
def mean_involved_electrodes(burst_time_intervals, df_recording):
    """Calculate mean number of electrodes involved in detected network bursts."""
    involved_electrodes = []
    for burst in burst_time_intervals:
        df_filtered = df_recording[(df_recording["Time"] >= burst[0]) & (df_recording["Time"] <= burst[1])]
        num_electrodes = df_filtered["Channel"].nunique()
        involved_electrodes.append(num_electrodes)

    mean_involved = np.mean(involved_electrodes)
    return mean_involved

def network_burst_mean_duration(burst_time_intervals):
    """Return the mean duration of detected network bursts."""
    durations = [burst[1] - burst[0] for burst in burst_time_intervals]
    return np.mean(durations) if durations else 0


def network_burst_freq_per_minute(burst_time_intervals, total_recording_time=30 * 60):
    """Calculate network burst frequency (bursts per minute) over the recording."""
    if len(burst_time_intervals) == 0:
        return 0
    return len(burst_time_intervals) / (total_recording_time) * 60  


def inter_network_burst_intervals(burst_time_intervals):   # INBI
    """Compute inter-network burst intervals from burst start times."""
    if len(burst_time_intervals) < 2:
        return np.array([])
    return np.diff([burst[0] for burst in burst_time_intervals])


def cv_of_INBI(inbi):
    """Return coefficient of variation, mean, and std of inter-network burst intervals."""
    mean_interval = np.mean(inbi)
    std_interval = np.std(inbi, ddof=1)  # sample std
    cv = std_interval / mean_interval
    return cv, mean_interval, std_interval


def fraction_in_burst(burst_time_intervals, total_recording_duration=30*60):
    """Calculate fraction and total duration of recording spent in bursts."""
    burst_durations = burst_time_intervals[:, 1] - burst_time_intervals[:, 0]
    total_burst_time = np.sum(burst_durations)
    fnb = total_burst_time / total_recording_duration       
    return fnb, total_burst_time


def mean_ISI_within_burst(burst_time_intervals, df_recording):
    """Compute mean inter-spike interval (ms) for spikes occurring within bursts."""
    ISI_list = []
    for burst in burst_time_intervals:
        spikes_within_burst = df_recording[(df_recording["Time"] >= burst[0]) & (df_recording["Time"] <= burst[1])]
        isi_within_burst = np.diff(spikes_within_burst["Time"]) * 1000  # Convert to ms
        mean_isi_within_burst = np.mean(isi_within_burst) if len(isi_within_burst) > 0 else 0
        ISI_list.append(mean_isi_within_burst)

    return np.mean(ISI_list) if ISI_list else 0


def mean_ISI_outside_burst(bursts_times, df_):
    """Compute mean inter-spike interval (ms) for spikes occurring outside bursts."""
    outside_ISI = []
    burst_intervals = [(burst[0], burst[1]) for burst in bursts_times]
    burst_intervals.sort()
    
    if burst_intervals:
        if burst_intervals[0][0] > 0:
            outside_intervals = [(0, burst_intervals[0][0])]
        else:
            outside_intervals = []
        
        for i in range(len(burst_intervals) - 1):
            outside_intervals.append((burst_intervals[i][1], burst_intervals[i + 1][0]))
        
        outside_intervals.append((burst_intervals[-1][1], df_["Time"].max()))
    else:
        outside_intervals = [(0, df_["Time"].max())]
    
    for interval in outside_intervals:
        df_filtered = df_[(df_["Time"] >= interval[0]) & (df_["Time"] <= interval[1])]
        isi_outside_burst = np.diff(df_filtered["Time"]) * 1000  # Convert to ms
        outside_ISI.extend(isi_outside_burst)
    
    return np.mean(outside_ISI) if outside_ISI else 0

In [4]:
recording_baseline = pd.read_csv("./data/spikes_recording_baseline.csv")
recording_baseline

,Channel,Time
0,E8_33,0.00152
1,C8_34,0.00160
2,B5_24,0.00184
3,C8_43,0.00200
4,F6_14,0.00240
...,...,...
3270729,D5_44,1799.99680
3270730,B6_12,1799.99712
3270731,E7_23,1799.99792
3270732,F5_21,1799.99792


## Baseline Features

In [5]:
recording_per_channel_baseline = pd.read_pickle("./data/spikes_baseline_processed.pkl")
recording_per_channel_baseline

,Channel,Time,well,channel,spike_count
0,A5_11,"[0.13936, 0.46488, 0.7532, 0.97168, 1.2196, 1....",A5,11,19572
1,A5_12,"[0.30912, 1.05888, 1.46736, 2.16472, 2.9056, 2...",A5,12,9228
2,A5_13,"[0.24208, 0.33024, 0.68032, 1.44944, 3.71648, ...",A5,13,5365
3,A5_14,"[0.42488, 1.37192, 1.40024, 1.62248, 1.7092, 1...",A5,14,23267
4,A5_21,"[0.64, 0.73336, 1.32408, 1.6256, 2.45096, 2.62...",A5,21,11932
...,...,...,...,...,...
379,F8_34,"[1.40888, 1.95128, 2.24448, 2.58248, 2.61496, ...",F8,34,9202
380,F8_41,"[0.41984, 0.47576, 0.5532, 1.60848, 2.23488, 2...",F8,41,1691
381,F8_42,"[0.92936, 2.122, 2.24208, 2.30432, 3.20448, 3....",F8,42,7138
382,F8_43,"[10.2164, 12.16048, 26.43208, 32.93592, 47.397...",F8,43,309


**Note:** `recording_per_channel` is processed - has active and non-noisy channels only.

In [6]:
recording_baseline = recording_baseline[recording_baseline["Channel"].isin(recording_per_channel_baseline["Channel"])]
recording_baseline

,Channel,Time
0,E8_33,0.00152
1,C8_34,0.00160
2,B5_24,0.00184
3,C8_43,0.00200
4,F6_14,0.00240
...,...,...
3270729,D5_44,1799.99680
3270730,B6_12,1799.99712
3270731,E7_23,1799.99792
3270732,F5_21,1799.99792


# Feature Extraction - Loop over all baseline wells

In [7]:
wells = ["A5", "A6", "A7", "A8", 
         "B5", "B6", "B7", "B8", 
         "C5", "C6", "C7", "C8", 
         "D5", "D6", "D7", "D8", 
         "E5", "E6", "E7", "E8",
         "F5", "F6", "F7", "F8"]

In [8]:
threshold_dict_baseline = {
    "A5": 100, "A6": 100, "A7": 100, "A8": 100,
    "B5": 200, "B6": 100, "B7": 100, "B8": 100,
    "C5": 100, "C6": 100, "C7": 100, "C8": 100,
    "D5": 100, "D6": 100, "D7": 100, "D8": 100,
    "E5": 100, "E6": 100, "E7": 100, "E8": 100,
    "F5": 150, "F6": 150, "F7": 150, "F8": 150,
}  

In [9]:
results = []

for well in wells:
    df_well = recording_baseline[recording_baseline["Channel"].str.startswith(f"{well}_")]
    bursts = BurstDetector.detect(spiketime=df_well.Time, n=10, threshold=threshold_dict_baseline[well])

    filtered_bursts = filter_bursts_by_min_involved_electrodes(bursts, df_well, min_electrodes=3)
    
    filtered_bursts = np.array(filtered_bursts)
    
    bursts_mean_duration = network_burst_mean_duration(filtered_bursts)

    mean_involved_elec = mean_involved_electrodes(filtered_bursts, df_well)  # Optional, can be ignored
    burst_frequency_per_minute = network_burst_freq_per_minute(filtered_bursts)
    INBI = inter_network_burst_intervals(filtered_bursts)
    cv, mean_inbi, std_inbi = cv_of_INBI(INBI)

    fnb, total_burst_time = fraction_in_burst(filtered_bursts)

    mean_ISI_within_burst_well = mean_ISI_within_burst(filtered_bursts, df_well)
    mean_ISI_outside_burst_well = mean_ISI_outside_burst(filtered_bursts, df_well)

    results.append({
        "well": well,
        "bursts_mean_duration_baseline": bursts_mean_duration,
        "burst_freq_baseline": burst_frequency_per_minute,
        "INBI_baseline": mean_inbi,
        "cv_of_INBI_baseline": cv,
        "fnb_baseline": fnb,
        "ISI_within_baseline": mean_ISI_within_burst_well,
        "ISI_outside_baseline": mean_ISI_outside_burst_well,
        "involved_electrodes_baseline": mean_involved_elec,  # Optional, can be ignored
    })


In [10]:
results_df = pd.DataFrame(results)
results_df

,well,bursts_mean_duration_baseline,burst_freq_baseline,INBI_baseline,cv_of_INBI_baseline,fnb_baseline,ISI_within_baseline,ISI_outside_baseline,involved_electrodes_baseline
0,A5,0.736059,18.500000,3.223904,1.964713,0.226951,9.541680,34.720104,8.686486
1,A6,1.109586,11.433333,5.170540,2.033249,0.211438,9.333734,47.510547,9.189504
2,A7,0.657318,25.900000,2.305674,1.704887,0.283742,9.772066,31.961822,6.987130
3,A8,1.003370,10.266667,5.781035,1.773971,0.171688,9.268820,46.466563,8.941558
4,B5,3.193089,7.233333,8.299698,0.795031,0.384945,12.013411,88.757704,10.967742
5,B6,1.525736,7.966667,7.546302,1.427138,0.202584,8.362839,57.324257,9.757322
6,B7,1.589892,10.200000,5.892006,1.393502,0.270282,8.624389,47.911073,7.718954
7,B8,1.626026,11.000000,5.413796,1.467835,0.298105,8.746902,53.412817,6.163636
8,C5,1.213061,13.566667,4.407533,1.773620,0.274287,9.173268,43.404870,8.321867
9,C6,1.492524,6.966667,8.497630,1.495327,0.173299,8.441072,59.102521,10.234450


In [11]:
results_df.to_csv("./data/processed/network_features_baseline.csv", index=False)

## Treatment Features

In [12]:
recording_treatment = pd.read_csv("./data/spikes_recording_treatment.csv")
recording_treatment

,Channel,Time
0,B7_13,0.00144
1,C6_23,0.00160
2,E5_21,0.00432
3,B8_11,0.00464
4,B8_11,0.00464
...,...,...
2288055,B6_14,1799.74712
2288056,F5_23,1799.74760
2288057,A5_11,1799.74768
2288058,A5_11,1799.74768


In [13]:
recording_per_channel_treatment = pd.read_pickle("./data/spikes_treatment_processed.pkl")
recording_per_channel_treatment

,Channel,Time,well,channel,spike_count
0,A5_11,"[0.04968, 0.26144, 0.2876, 0.33344, 0.5552, 0....",A5,11,22684
1,A5_12,"[0.63064, 1.71168, 2.22104, 2.56928, 3.32376, ...",A5,12,14626
2,A5_13,"[3.52592, 4.60304, 5.43824, 7.73712, 9.6256, 1...",A5,13,7603
3,A5_14,"[0.7556, 1.30216, 1.85928, 1.89608, 1.89616, 1...",A5,14,25741
4,A5_21,"[2.59168, 3.10944, 3.1696, 4.8064, 6.33776, 6....",A5,21,14167
...,...,...,...,...,...
378,F8_33,"[5.20664, 24.64752, 33.44064, 36.32048, 37.405...",F8,33,1292
379,F8_34,"[3.7468, 14.6104, 18.22576, 19.27168, 25.01208...",F8,34,1218
380,F8_41,"[5.06896, 6.6676, 17.042, 19.21648, 23.71888, ...",F8,41,1262
381,F8_42,"[0.02592, 0.32912, 0.49104, 0.62184, 0.71512, ...",F8,42,3279


**Note:** `recording_per_channel` is processed - has active and non-noisy channels only.

In [14]:
recording_treatment = recording_treatment[recording_treatment["Channel"].isin(recording_per_channel_treatment["Channel"])]
recording_treatment

,Channel,Time
0,B7_13,0.00144
1,C6_23,0.00160
2,E5_21,0.00432
3,B8_11,0.00464
4,B8_11,0.00464
...,...,...
2288055,B6_14,1799.74712
2288056,F5_23,1799.74760
2288057,A5_11,1799.74768
2288058,A5_11,1799.74768


Overview of treatment conditions:

| Class | Label | Condition    | Description               | Well  |
|:-----:|:-----:|:-------------|:----------------|:----------------|
| 0     | 0     | Control      | Healthy                   | C5-C8 |
| 1     | 1     | Gabazine     | Hyper                     | A5-A8 |
| 2     | 1     | Kainic Acid  | Hyper                     | B5-B8 |
| 4     | 2     | GABA         | Hypo                      | D5-D8 |
| 5     | 2     | CNQX         | Hypo                      | E5-E8 |
| 6     | 2     | D-AP5        | Hypo                      | F5-F8 |

In [15]:
classnames = [1,1,1,1,2,2,2,2,0,0,0,0,4,4,4,4,5,5,5,5,6,6,6,6]
wells = ["A5", "A6", "A7", "A8", 
         "B5", "B6", "B7", "B8", 
         "C5", "C6", "C7", "C8", 
         "D5", "D6", "D7", "D8", 
         "E5", "E6", "E7", "E8",
         "F5", "F6", "F7", "F8"]

treatments = ["Gabazine", "Gabazine", "Gabazine", "Gabazine",
            "KainicAcid", "KainicAcid", "KainicAcid", "KainicAcid",
            "Control", "Control", "Control", "Control",
            "GABA", "GABA", "GABA", "GABA",
            "CNQX", "CNQX", "CNQX", "CNQX",
            "D-AP5", "D-AP5", "D-AP5", "D-AP5"]

labels = [1, 1, 1, 1, 
          1, 1, 1, 1, 
          0, 0, 0, 0, 
          2, 2, 2, 2, 
          2, 2, 2, 2, 
          2, 2, 2, 2]

In [16]:
threshold_dict_treatment = {
    "A5": 100, "A6": 100, "A7": 100, "A8": 100,
    "B5": 160, "B6": 160, "B7": 160, "B8": 160,
    "C5": 100, "C6": 100, "C7": 100, "C8": 100,
    "D5": 220, "D6": 220, "D7": 220, "D8": 220,
    "E5": 400, "E6": 400, "E7": 300, "E8": 300,
    "F5": 500, "F6": 300, "F7": 400, "F8": 400,
}

In [17]:
results = []

for i, well in enumerate(wells):
    df_well = recording_treatment[recording_treatment["Channel"].str.startswith(f"{well}_")]
    bursts = BurstDetector.detect(spiketime=df_well.Time, n=10, threshold=threshold_dict_treatment[well])

    filtered_bursts = filter_bursts_by_min_involved_electrodes(bursts, df_well, min_electrodes=3)
    
    filtered_bursts = np.array(filtered_bursts)
    
    bursts_mean_duration = network_burst_mean_duration(filtered_bursts)

    mean_involved_elec = mean_involved_electrodes(filtered_bursts, df_well)  # Optional, can be ignored
    burst_frequency_per_minute = network_burst_freq_per_minute(filtered_bursts)
    INBI = inter_network_burst_intervals(filtered_bursts)
    cv, mean_inbi, std_inbi = cv_of_INBI(INBI)

    fnb, total_burst_time = fraction_in_burst(filtered_bursts)

    mean_ISI_within_burst_well = mean_ISI_within_burst(filtered_bursts, df_well)
    mean_ISI_outside_burst_well = mean_ISI_outside_burst(filtered_bursts, df_well)

    results.append({
        "well": well,
        "treatment": treatments[i],
        "bursts_mean_duration": bursts_mean_duration,
        "burst_freq": burst_frequency_per_minute,
        "INBI": mean_inbi,
        "cv_of_INBI": cv,
        "fnb": fnb,
        "ISI_within": mean_ISI_within_burst_well,
        "ISI_outside": mean_ISI_outside_burst_well,
        "involved_electrodes": mean_involved_elec,  # Optional, can be ignored
    })

In [18]:
results_df = pd.DataFrame(results)
results_df

,well,treatment,bursts_mean_duration,burst_freq,INBI,cv_of_INBI,fnb,ISI_within,ISI_outside,involved_electrodes
0,A5,Gabazine,1.330969,7.166667,8.362693,1.652925,0.158977,8.260752,55.252790,9.037209
1,A6,Gabazine,2.728262,3.066667,19.124589,1.040919,0.139444,5.577697,91.821564,11.847826
2,A7,Gabazine,1.144516,11.966667,4.954291,1.510580,0.228267,8.717858,44.370328,7.278552
3,A8,Gabazine,2.184355,3.633333,16.625970,1.162786,0.132275,7.017007,70.886011,10.311927
4,B5,KainicAcid,0.468787,78.033333,0.769039,0.524098,0.609683,16.190530,32.374399,7.265271
5,B6,KainicAcid,0.548267,61.800000,0.969878,0.778393,0.564715,16.023357,33.718851,7.466019
6,B7,KainicAcid,1.010572,46.066667,1.298226,1.117266,0.775895,15.515767,33.523978,7.164978
7,B8,KainicAcid,1.409358,36.700000,1.624525,0.892338,0.862057,15.069276,32.938614,6.129882
8,C5,Control,0.916483,16.200000,3.684018,2.223902,0.247450,9.346791,42.411369,7.948560
9,C6,Control,1.877206,4.266667,14.018927,1.431214,0.133490,8.166162,65.584254,10.171875


In [19]:
results_df.to_csv("./data/processed/network_features_treatment.csv", index=False)